# Spam Email Classification with PyTorch Lightning

This notebook is the primary local classroom demo. It shows the full workflow:

- Kaggle download
- robust preprocessing
- `LightningDataModule`
- `LightningModule`
- `Trainer`, callbacks, and logging
- test evaluation
- single-text prediction
- artifact saving

## Local environment note

Create and activate a local Python 3.10+ virtual environment before running the notebook. The notebook does not create a venv for you; it only installs packages into the currently active environment.

In [1]:
from pathlib import Path
import subprocess
import sys

cwd = Path.cwd().resolve()
candidates = [cwd / 'requirements.txt', cwd.parent / 'requirements.txt']
requirements_path = next((path for path in candidates if path.exists()), None)
if requirements_path is None:
    raise FileNotFoundError('Could not find requirements.txt from the current notebook working directory.')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-r', str(requirements_path)])
print(f'Installed requirements from {requirements_path}')

Installed requirements from /home/weijiesun/cmput/intd-491-ML-Demo/requirements.txt


In [2]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / 'src').exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / 'src').exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError('Could not locate the repo root containing src/.')

SRC_DIR = REPO_ROOT / 'src'
SCRIPTS_DIR = REPO_ROOT / 'scripts'
for path in (SRC_DIR, SCRIPTS_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f'REPO_ROOT = {REPO_ROOT}')

REPO_ROOT = /home/weijiesun/cmput/intd-491-ML-Demo


## Kaggle token setup

Supported options:

1. Put `kaggle.json` at `~/.kaggle/kaggle.json` and run `chmod 600 ~/.kaggle/kaggle.json`
2. Upload `kaggle.json` below, and the helper will copy it into `~/.kaggle/` and apply `chmod 600`

In [3]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept='.json', multiple=False)
install_button = widgets.Button(description='Install kaggle.json', button_style='primary')
upload_output = widgets.Output()

def install_kaggle_token(_: object) -> None:
    with upload_output:
        upload_output.clear_output()
        if not uploader.value:
            print('No file uploaded. Skip this step if ~/.kaggle/kaggle.json already exists.')
            return
        upload = next(iter(uploader.value.values()))
        kaggle_dir = Path.home() / '.kaggle'
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        kaggle_path = kaggle_dir / 'kaggle.json'
        kaggle_path.write_bytes(upload['content'])
        kaggle_path.chmod(0o600)
        print(f'Saved Kaggle token to {kaggle_path}')

install_button.on_click(install_kaggle_token)
display(widgets.VBox([uploader, install_button, upload_output]))

In [4]:
from spam_lightning.config import ProjectConfig
from spam_lightning.utils.paths import ensure_dir

defaults = ProjectConfig()
KAGGLE_DATASET_SLUG = 'harshsinha1234/email-spam-classification'
RAW_DIR = ensure_dir(REPO_ROOT / defaults.paths.raw_dir)
PROCESSED_DIR = ensure_dir(REPO_ROOT / defaults.paths.processed_dir)
ARTIFACTS_DIR = ensure_dir(REPO_ROOT / defaults.paths.artifacts_dir)
LOGS_DIR = ensure_dir(REPO_ROOT / defaults.paths.logs_dir)
LABEL_MAP_OVERRIDE = {}
SEED = defaults.preprocess.seed

# FIXED: print now reflects the actual slug instead of a misleading reminder
print(f'Using KAGGLE_DATASET_SLUG = {KAGGLE_DATASET_SLUG}')
print(f'RAW_DIR = {RAW_DIR}')
print(f'PROCESSED_DIR = {PROCESSED_DIR}')

Using KAGGLE_DATASET_SLUG = harshsinha1234/email-spam-classification
RAW_DIR = /home/weijiesun/cmput/intd-491-ML-Demo/data/raw
PROCESSED_DIR = /home/weijiesun/cmput/intd-491-ML-Demo/data/processed


In [5]:
import sys

# FIXED: Guard module cache-busting with a flag; set to False in production
_FORCE_RELOAD = True
if _FORCE_RELOAD:
    for name in list(sys.modules):
        if name == "spam_lightning" or name.startswith("spam_lightning."):
            del sys.modules[name]

# FIXED: Removed unused `discover_tabular_files` import (used in the next cell)
from download_kaggle import download_dataset

tabular_files = download_dataset(dataset=KAGGLE_DATASET_SLUG, raw_dir=RAW_DIR)
tabular_files

2026-03-03 21:55:38,618 | INFO | spam_lightning | Downloading Kaggle dataset 'harshsinha1234/email-spam-classification' into /home/weijiesun/cmput/intd-491-ML-Demo/data/raw/harshsinha1234__email-spam-classification


Dataset URL: https://www.kaggle.com/datasets/harshsinha1234/email-spam-classification


100%|██████████| 2.89M/2.89M [00:00<00:00, 460MB/s]
2026-03-03 21:55:40,010 | INFO | spam_lightning | Discovered 1 tabular files:
2026-03-03 21:55:40,011 | INFO | spam_lightning |   - /home/weijiesun/cmput/intd-491-ML-Demo/data/raw/harshsinha1234__email-spam-classification/emails.csv


[PosixPath('/home/weijiesun/cmput/intd-491-ML-Demo/data/raw/harshsinha1234__email-spam-classification/emails.csv')]

In [6]:
import pandas as pd
from spam_lightning.data.preprocessing import discover_tabular_files, select_input_file

all_tabular_files = discover_tabular_files(RAW_DIR)
selected_file = select_input_file(RAW_DIR, None)
if selected_file.suffix.lower() == '.tsv':
    preview_df = pd.read_csv(selected_file, sep='	', nrows=5)
else:
    preview_df = pd.read_csv(selected_file, nrows=5)

print('Discovered files:')
for path in all_tabular_files:
    print(f' - {path}')
print(f'Selected file: {selected_file}')
print(f'Columns: {preview_df.columns.tolist()}')
preview_df = preview_df[['text', 'spam']]
preview_df.head()

Discovered files:
 - /home/weijiesun/cmput/intd-491-ML-Demo/data/raw/harshsinha1234__email-spam-classification/emails.csv
Selected file: /home/weijiesun/cmput/intd-491-ML-Demo/data/raw/harshsinha1234__email-spam-classification/emails.csv
Columns: ['text', 'spam', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24', 'Unnamed: 25', 'Unnamed: 26', 'Unnamed: 27', 'Unnamed: 28', 'Unnamed: 29', 'Unnamed: 30', 'Unnamed: 31', 'Unnamed: 32', 'Unnamed: 33', 'Unnamed: 34', 'Unnamed: 35', 'Unnamed: 36', 'Unnamed: 37', 'Unnamed: 38', 'Unnamed: 39', 'Unnamed: 40', 'Unnamed: 41', 'Unnamed: 42', 'Unnamed: 43', 'Unnamed: 44', 'Unnamed: 45', 'Unnamed: 46', 'Unnamed: 47', 'Unnamed: 48', 'Unnamed: 49', 'Unnamed: 50', 'Unnamed:

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [7]:
preview_df = pd.read_csv(selected_file)
preview_df['spam'].value_counts()

spam
0                                                                                               4359
1                                                                                               1368
 its termination would not  have such a phenomenal impact on the power situation .  however        1
 mr suresh prabhu                                                                                  1
Name: count, dtype: int64

In [8]:
from spam_lightning.data.preprocessing import preprocess_dataset

preprocess_result = preprocess_dataset(
    raw_dir=RAW_DIR,
    out_dir=PROCESSED_DIR,
    label_map=LABEL_MAP_OVERRIDE or None,
    label_col='spam',
    seed=SEED,
    dataset_slug=KAGGLE_DATASET_SLUG,
)
preprocess_result

2026-03-03 21:55:40,512 | WARNING | spam_lightning | Dropped 4 rows with unmappable label values: [nan, ' its termination would not  have such a phenomenal impact on the power situation .  however ', ' mr suresh prabhu ']


{'stats': {'source_file': '/home/weijiesun/cmput/intd-491-ML-Demo/data/raw/harshsinha1234__email-spam-classification/emails.csv',
  'text_col': 'text',
  'label_col': 'spam',
  'label_mapping': {'0': 0,
   '0.0': 0,
   '1': 1,
   '1.0': 1,
   'false': 0,
   'ham': 0,
   'legit': 0,
   'negative': 0,
   'no': 0,
   'nonspam': 0,
   'notspam': 0,
   'safe': 0,
   'true': 1,
   'junk': 1,
   'positive': 1,
   'spam': 1,
   'yes': 1},
  'lowercase': True,
  'seed': 42,
  'total_rows': 5727,
  'split_ratios': {'train': 0.7, 'val': 0.15, 'test': 0.15},
  'split_sizes': {'train': 4008, 'val': 859, 'test': 860},
  'spam_ratio': {'train': 0.23877245508982037,
   'val': 0.23864959254947612,
   'test': 0.23953488372093024}},
 'files': {'train': '/home/weijiesun/cmput/intd-491-ML-Demo/data/processed/train.csv',
  'val': '/home/weijiesun/cmput/intd-491-ML-Demo/data/processed/val.csv',
  'test': '/home/weijiesun/cmput/intd-491-ML-Demo/data/processed/test.csv',
  'stats': '/home/weijiesun/cmput/intd-

In [9]:
import json
import pandas as pd
from IPython.display import display

stats = json.loads((PROCESSED_DIR / 'stats.json').read_text(encoding='utf-8'))
print(json.dumps(stats, indent=2))

ratio_df = pd.DataFrame(
    {
        'split': list(stats['spam_ratio'].keys()),
        'spam_ratio': list(stats['spam_ratio'].values()),
        'count': [stats['split_sizes'][split] for split in stats['spam_ratio'].keys()],
    }
)
display(ratio_df)

{
  "source_file": "/home/weijiesun/cmput/intd-491-ML-Demo/data/raw/harshsinha1234__email-spam-classification/emails.csv",
  "text_col": "text",
  "label_col": "spam",
  "label_mapping": {
    "0": 0,
    "0.0": 0,
    "1": 1,
    "1.0": 1,
    "false": 0,
    "ham": 0,
    "legit": 0,
    "negative": 0,
    "no": 0,
    "nonspam": 0,
    "notspam": 0,
    "safe": 0,
    "true": 1,
    "junk": 1,
    "positive": 1,
    "spam": 1,
    "yes": 1
  },
  "lowercase": true,
  "seed": 42,
  "total_rows": 5727,
  "split_ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "split_sizes": {
    "train": 4008,
    "val": 859,
    "test": 860
  },
  "spam_ratio": {
    "train": 0.23877245508982037,
    "val": 0.23864959254947612,
    "test": 0.23953488372093024
  }
}


,split,spam_ratio,count
0,train,0.238772,4008
1,val,0.238650,859
2,test,0.239535,860


In [10]:
from spam_lightning.data.datamodule import SpamDataModule

datamodule = SpamDataModule(
    data_dir=PROCESSED_DIR,
    batch_size=defaults.data.batch_size,
    num_workers=0,
    pin_memory=False,
    lowercase=stats['lowercase'],
    min_freq=defaults.data.min_freq,
    max_vocab_size=defaults.data.max_vocab_size,
)
datamodule.setup('fit')
print(f'Vocab size: {datamodule.vocab_size}')

Vocab size: 18339


In [11]:
tokens, offsets, labels = next(iter(datamodule.train_dataloader()))
print(f'tokens.shape = {tuple(tokens.shape)}')
print(f'offsets.shape = {tuple(offsets.shape)}')
print(f'labels.shape = {tuple(labels.shape)}')
print('First offsets:', offsets[:10].tolist())

tokens.shape = (16949,)
offsets.shape = (64,)
labels.shape = (64,)
First offsets: [0, 63, 296, 398, 457, 844, 953, 1114, 1276, 1334]


In [12]:
import json
import pytorch_lightning as L
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor, ModelCheckpoint
from pytorch_lightning.loggers import CSVLogger

from spam_lightning.models.lit_model import SpamLitModule
from spam_lightning.utils.seed import set_global_seed

set_global_seed(SEED)
vocab_path = ARTIFACTS_DIR / 'vocab.json'
datamodule.save_vocab(vocab_path)

model = SpamLitModule(
    vocab_size=datamodule.vocab_size,
    embed_dim=defaults.model.embed_dim,
    learning_rate=defaults.model.learning_rate,
    pad_index=datamodule.pad_index,
)
checkpoint_callback = ModelCheckpoint(
    monitor='val_f1',
    mode='max',
    save_top_k=1,
    filename='best',
    dirpath=str(ARTIFACTS_DIR),
)
trainer = L.Trainer(
    accelerator='auto',
    devices='auto',
    max_epochs=defaults.train.max_epochs,
    deterministic=True,
    precision='32-true',
    callbacks=[
        checkpoint_callback,
        EarlyStopping(monitor='val_f1', mode='max', patience=3),
        LearningRateMonitor(logging_interval='epoch'),
    ],
    logger=CSVLogger(save_dir=str(LOGS_DIR), name='spam_lightning'),
)
trainer.fit(model, datamodule=datamodule)

# FIXED: Guard against training failure that produces no checkpoint
BEST_CKPT_PATH = checkpoint_callback.best_model_path
if not BEST_CKPT_PATH:
    raise RuntimeError('Training did not produce a checkpoint. Check training logs.')

config_payload = {
    'dataset_slug': KAGGLE_DATASET_SLUG,
    'selected_raw_file': stats.get('source_file'),
    'text_col': stats.get('text_col'),
    'label_col': stats.get('label_col'),
    'label_mapping': stats.get('label_mapping', {}),
    'split_ratios': stats.get('split_ratios'),
    'seed': SEED,
    'lowercase': stats.get('lowercase', True),
    'vocab': {
        'min_freq': defaults.data.min_freq,
        'max_vocab_size': defaults.data.max_vocab_size,
        'vocab_path': str(vocab_path.resolve()),
    },
    'model': {
        'embed_dim': defaults.model.embed_dim,
        'learning_rate': defaults.model.learning_rate,
        'vocab_size': datamodule.vocab_size,
        'pad_index': datamodule.pad_index,
    },
    'trainer': {
        'max_epochs': defaults.train.max_epochs,
        'precision': '32-true',
        'deterministic': True,
        'batch_size': defaults.data.batch_size,
        'num_workers': 0,
        'pin_memory': False,
    },
    'artifacts': {
        'best_ckpt': BEST_CKPT_PATH,
        'vocab_json': str(vocab_path.resolve()),
        'config_json': str((ARTIFACTS_DIR / 'config.json').resolve()),
    },
}
(ARTIFACTS_DIR / 'config.json').write_text(json.dumps(config_payload, indent=2), encoding='utf-8')
print(f'Best checkpoint path: {BEST_CKPT_PATH}')

Seed set to 42
Trainer will use only 1 of 8 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=8)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
/home/weijiesun/anaconda3/envs/tabular_ml_env/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:881: Checkpoint directory /home/weijiesun/cmput/intd-491-ML-Demo/artifacts exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]

  | Name       | Type              | Params | Mode  | FLOPs
-------------------------------------------------------

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/weijiesun/anaconda3/envs/tabular_ml_env/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.
/home/weijiesun/anaconda3/envs/tabular_ml_env/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_epochs=5` reached.


Best checkpoint path: /home/weijiesun/cmput/intd-491-ML-Demo/artifacts/best.ckpt


In [13]:
BEST_CKPT_PATH

'/home/weijiesun/cmput/intd-491-ML-Demo/artifacts/best.ckpt'

In [14]:
test_results = trainer.test(model, datamodule=datamodule, ckpt_path='best')
test_results

Restoring states from the checkpoint path at /home/weijiesun/cmput/intd-491-ML-Demo/artifacts/best.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3,4,5,6,7]
Loaded model weights from the checkpoint at /home/weijiesun/cmput/intd-491-ML-Demo/artifacts/best.ckpt
/home/weijiesun/anaconda3/envs/tabular_ml_env/lib/python3.10/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'test_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=95` in the `DataLoader` to improve performance.


Testing: |          | 0/? [00:00<?, ?it/s]

────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
       Test metric             DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
        test_acc            0.9046511650085449
         test_f1            0.7602339386940002
        test_loss           0.24499468505382538
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'test_loss': 0.24499468505382538,
  'test_acc': 0.9046511650085449,
  'test_f1': 0.7602339386940002}]

In [16]:
import torch
from spam_lightning.data.preprocessing import clean_text
from spam_lightning.data.text_utils import load_vocab, regex_tokenize
from spam_lightning.models.lit_model import SpamLitModule

sample_text = 'free money!!! click now to claim your prize'
best_model = SpamLitModule.load_from_checkpoint(BEST_CKPT_PATH)
best_model.eval()
vocab = load_vocab(ARTIFACTS_DIR / 'vocab.json')
normalized_text = clean_text(sample_text, lowercase=bool(stats['lowercase']))
token_ids = [vocab.lookup_index(token) for token in (regex_tokenize(normalized_text, lowercase=False) or ['<unk>'])]

# FIXED: Validate tokenisation produced usable ids before inference
if not token_ids:
    raise ValueError(f'Tokenization produced no tokens for input: {sample_text!r}')

with torch.no_grad():
    probability = torch.sigmoid(
        best_model(
            torch.tensor(token_ids, dtype=torch.long).to(best_model.device),
            torch.tensor([0], dtype=torch.long).to(best_model.device),
        )
    ).item()

predicted_label = 'spam' if probability >= 0.5 else 'ham'
print(f'Text: {sample_text}')
print(f'Spam probability: {probability:.4f}')
print(f'Predicted label: {predicted_label}')

Text: free money!!! click now to claim your prize
Spam probability: 0.9834
Predicted label: spam


In [17]:
print('Artifacts:')
for path in sorted(ARTIFACTS_DIR.glob('*')):
    print(f' - {path.name}')

Artifacts:
 - best.ckpt
 - config.json
 - vocab.json
